# Notebook 02 v4 — Certified Expanded Point-in-Time historical Source Panel

**Sequential Pipeline Overhaul — Step 2 of 5 (`Notebook 02`).**
This notebook integrates our Step 1 Certified Universe (`universe_security_master_v5_CERTIFIED.parquet` and `candidate_membership_v5_CERTIFIED.parquet`) to construct the final Point-in-Time (PIT) source panel for January 2016–December 2019, maintaining strict parity and preserving the repaired 2020+ snapshot.

### Core Enhancements:
1. **Verifiable Data Lineage:** Imports historical metadata from the `Notebook 01 v5` verified run directories.
2. **Dynamic Identity & CIK Alignments:** Incorporates the `admissible_for_nb02_history_2016_2019` flag and mapped CIK IDs, resolving dual-class share alignments using `canonical_issuer_id` instead of 8 manually-hardcoded overrides.
3. **Advanced point-In-Time Availability Verification:** Applies trading-session sequence filters on raw balance-sheet, income-statement, and cash-flow data. This maps filed information to the `tradable_from` next-market session based strictly on calendar constraints (ensuring publication availability date is less than the session datetime, checking standard weekend or holiday gaps).
4. **Incorporation of Terminal Exit Chronologies:** Embeds exit classifications and exact terminal dates to eradicate future leakages.
5. **Robust Handoff Checkpoint Verification:** Operates 8 executable point-in-time checks, outputting `expanded_n2_contract.json` and a diagnostic visual bundle.

In [7]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import zipfile
import numpy as np
import pandas as pd
from IPython.display import display

VERSION = 'nb02_v4_HISTORY_2016_CERTIFIED'
HERE = Path.cwd().resolve()
PROJECT_ROOT = HERE.parent if HERE.name.lower() == 'notebooks' else HERE
DATA = PROJECT_ROOT / 'data'

# Search for the latest Notebook 01 v5 Certified runs
v5_audits_dir = DATA / 'interim/audits/nb01_history_v5_certified'
if not v5_audits_dir.is_dir():
    raise FileNotFoundError(f"Step 1 certified directory not found at: {v5_audits_dir}. Please run Notebook 01 v5 first.")
v5_runs = sorted([p for p in v5_audits_dir.iterdir() if p.is_dir() and (p / 'nb01_universe_certification_contract.json').exists()])
if not v5_runs:
    raise FileNotFoundError("Could not find any completed Step 1 Certified Runs.")
CERTIFIED_N1_DIR = v5_runs[-1]

# Baseline notebook 2 runs (N2)
BASELINE_DIR = DATA / 'interim/audits/nb02_v2/20261014T163316Z'
CACHE_ROOTS = [
    DATA / 'raw/history_expansion/historical_expansion_v2_20261019',
    DATA / 'raw/history_expansion/historical_reconciliation_v3',
    DATA / 'raw/history_expansion/historical_issuer_recovery_v4',
]
CALENDAR_PATH = DATA / 'raw/market/VOO_dividend_adjusted_2010_plus.parquet'
BASELINE_PRICE_PATH = DATA / 'interim/panel/equity_daily_prices_2018_plus.parquet'

MAX_STATEMENT_AGE_DAYS = 180
ACCEPT_PROVISIONAL_VENDOR_HISTORY = True

PINNED_HASHES = {
    'baseline_panel': '846de7d0ca2e06908940b23786bbbb21b5ffa8f925d3f48f05caa620c0773a47',
    'baseline_ledger': '183f268ec243d4d859973cefd08b04fbf0a38366e5e1593314df2f66a6d213c3',
}

INPUT_PATHS = {
    'baseline_panel': BASELINE_DIR / 'candidate_pit_source_panel.parquet',
    'baseline_ledger': BASELINE_DIR / 'candidate_statement_ledger.parquet',
    'certified_membership': CERTIFIED_N1_DIR / 'candidate_membership_v5_CERTIFIED.parquet',
    'certified_master': CERTIFIED_N1_DIR / 'universe_security_master_v5_CERTIFIED.parquet',
    'calendar': CALENDAR_PATH,
    'baseline_prices': BASELINE_PRICE_PATH,
}

def sha_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1024*1024), b''): h.update(chunk)
    return h.hexdigest()

print('Verified latest Step 1 Certified Directory:', CERTIFIED_N1_DIR.name)
print('Project ROOT:', PROJECT_ROOT)

Verified latest Step 1 Certified Directory: 20261030T143346756394Z
Project ROOT: /Users/neilyejjey/stock_signal_engine_v2


## 2. Reusable point-In-Time Pipeline Methods

Builds historical financial and EOD pricing grids utilizing next-session calendar and data timing constraints to avoid future observation overlaps.

In [8]:
# === SURGICAL PATCH FOR NB02 v4: ROW-LEVEL CIK CHECK & EXPLICIT PRICE OVERLAP CLASSIFICATION ===

def prepare_ledger(raw, sessions, security_master):
    l = raw.copy()
    l['date'] = l['date'].map(day)
    l = l.loc[l.date.between('2012-01-01', '2019-12-31')].copy()
    for c in ['acceptedDate', 'filingDate', 'fiscalYear', 'period', 'cik', 'symbol']:
        if c not in l: l[c] = None

    # Preserve row-level CIK from the raw statement payload before merging master metadata
    l['raw_row_cik'] = l['cik'].map(lambda x: str(x).strip()[:-2] if str(x).strip().endswith('.0') else str(x).strip()).map(
        lambda s: s.lstrip('0') if s.isdigit() else ''
    )

    l = l.merge(security_master[[
        'candidate_security_key',
        'resolved_historical_cik',
        'canonical_issuer_id',
        'identity_certification_status',
        'exit_classification',
        'documented_delisted_date',
        'price_coverage_status',
    ]], left_on='security_key', right_on='candidate_security_key', how='left')

    l['normalized_cik'] = l['raw_row_cik'].where(l['raw_row_cik'].ne(''), l['resolved_historical_cik'].fillna(''))
    l['identity_status'] = l['identity_certification_status'].fillna('missing_cik')

    # Quarantine any individual statement row whose row-level CIK conflicts with the certified master CIK
    master_cik = l['resolved_historical_cik'].fillna('')
    row_cik_mismatch = l['raw_row_cik'].ne('') & master_cik.ne('') & (l['raw_row_cik'] != master_cik)
    l.loc[row_cik_mismatch, 'identity_status'] = 'historical_cik_mismatch'
    l.loc[l['symbol'].notna() & l['symbol'].ne(l['security_key']), 'identity_status'] = 'response_symbol_mismatch'

    l['identity_usable'] = l['identity_status'].isin([
        'SINGLE_CONSISTENT_VENDOR_CIK_PROVISIONAL',
        'REGISTERED_SEC_AND_VENDOR_CIK_MATCH',
        'SEC_SUBMISSIONS_CACHE_CIK_MATCH',
    ])

    l['availability_date'] = pd.NaT
    l['tradable_from'] = pd.NaT
    l['timing_status'] = 'no_valid_publication_date'

    for idx, r in l.iterrows():
        dates = [day(r.get(c)) for c in ['acceptedDate', 'filingDate']]
        valid = [d for d in dates if pd.notna(d) and d > r.date]
        if not valid: continue
        available = max(valid)
        pos = sessions.searchsorted(available, side='right')
        l.at[idx, 'availability_date'] = available
        if available < sessions.min() or pos >= len(sessions): continue
        l.at[idx, 'tradable_from'] = sessions[pos]
        l.at[idx, 'timing_status'] = 'later_valid_vendor_date_next_session'

    l['quarter_usable'] = l.period.isin(['Q1', 'Q2', 'Q3', 'Q4'])
    l['pit_usable_timestamp'] = l.tradable_from.notna()
    l['availability_status'] = l.timing_status
    l['value_vintage_status'] = 'current_vendor_snapshot_original_vintage_unverified'
    l['source_scope'] = 'history_2016_2019'

    l = l.drop_duplicates(['security_key', 'statement_type', 'source_record_sha256']).reset_index(drop=True)
    l['ledger_row_id'] = l.index.astype('int64')
    l['duplicate_conflict'] = False
    timed = l.loc[l.pit_usable_timestamp]
    keys = ['security_key', 'statement_type', 'date', 'tradable_from']
    dupe = timed.duplicated(keys, keep=False)
    l.loc[timed.loc[dupe].index, 'duplicate_conflict'] = True
    l['dependency_usable'] = l.identity_usable & l.quarter_usable & l.pit_usable_timestamp & ~l.duplicate_conflict
    return l

def overlap_audit(new_prices, old_prices):
    """Separates genuine adjusted-price scale conflicts from insufficient overlap (<60 days)."""
    left = new_prices.loc[new_prices.price_usable, ['security_key', 'date', 'adjClose']]
    right = old_prices[['security_key', 'date', 'adjClose']].copy()
    right['date'] = pd.to_datetime(right.date)
    if right.duplicated(['security_key', 'date']).any():
        raise ValueError('Baseline daily prices have duplicate keys')
    matched = left.merge(right, on=['security_key', 'date'], suffixes=('_new', '_old'), validate='one_to_one')
    matched = matched.loc[matched.adjClose_old.gt(0) & np.isfinite(matched.adjClose_old)].copy()
    matched['ratio'] = matched.adjClose_old / matched.adjClose_new
    rows = []
    for key, g in matched.groupby('security_key'):
        ratio = float(g.ratio.median())
        deviation = float((g.ratio / ratio - 1.0).abs().max())
        n_ov = int(len(g))
        if n_ov >= 60 and deviation <= 0.001:
            ov_status = 'PASS_CONSTANT_SCALE'
        elif n_ov < 60 and deviation <= 0.001:
            ov_status = 'INSUFFICIENT_OVERLAP_LT_60_DAYS_SCALE_CONSISTENT'
        elif deviation > 0.001:
            ov_status = 'GENUINE_ADJUSTED_PRICE_CONFLICT'
        else:
            ov_status = 'UNCLASSIFIED_OVERLAP'
        rows.append(dict(
            security_key=key,
            overlap_rows=n_ov,
            median_old_to_new_scale=ratio,
            maximum_relative_scale_deviation=deviation,
            constant_scale_diagnostic_pass=bool(n_ov >= 60 and deviation <= 0.001),
            overlap_status=ov_status,
            automatic_stitch_approved=False,
        ))
    return pd.DataFrame(rows, columns=[
        'security_key', 'overlap_rows', 'median_old_to_new_scale',
        'maximum_relative_scale_deviation', 'constant_scale_diagnostic_pass',
        'overlap_status', 'automatic_stitch_approved'
    ])

## 3. Verify Inputs & Load Cached Point-in-Time Data

Loads and verifies historical membership, master registers, and pricing arrays.

In [9]:
missing = [str(p) for p in INPUT_PATHS.values() if not p.is_file()]
missing += [str(p) for p in CACHE_ROOTS if not p.is_dir()]
if missing:
    raise FileNotFoundError('Check PROJECT_ROOT and pinned run/cache paths:\n'+'\n'.join(missing))

input_hashes = {k: sha_file(p) for k, p in INPUT_PATHS.items() if k != 'certified_master'} # dynamic validation
baseline_panel_df = pd.read_parquet(INPUT_PATHS['baseline_panel'])
baseline_ledger_df = pd.read_parquet(INPUT_PATHS['baseline_ledger'])
baseline_prices_df = pd.read_parquet(BASELINE_PRICE_PATH)

membership_df = pd.read_parquet(INPUT_PATHS['certified_membership'])
security_master_df = pd.read_parquet(INPUT_PATHS['certified_master'])
voo_df = pd.read_parquet(CALENDAR_PATH)

calendar_dates = pd.to_datetime(voo_df['date'], errors='raise')
if calendar_dates.isna().any() or calendar_dates.duplicated().any(): raise ValueError('Invalid observed VOO calendar')
sessions = pd.DatetimeIndex(calendar_dates.sort_values().unique())

if sessions.min() > pd.Timestamp('2014-01-01') or sessions.max() < pd.Timestamp('2020-01-01'):
    raise ValueError('Calendar does not cover historical warmup and boundary')
if pd.to_datetime(baseline_panel_df.as_of_date).min() < pd.Timestamp('2020-01-01'):
    raise ValueError('Baseline overlaps historical extension')
if baseline_panel_df.duplicated(['security_key', 'as_of_date']).any(): raise ValueError('Baseline panel duplicate keys')

monthly_history_base_df = monthly_base(membership_df, sessions)
symbols = set(monthly_history_base_df.query_symbol.dropna()) - {''}

raw_statements_df, raw_prices_df, cache_audit_df = read_caches(CACHE_ROOTS, symbols)
print('Historical company-months:', len(monthly_history_base_df))
print('Cached statement / price records:', len(raw_statements_df), len(raw_prices_df))
display(cache_audit_df.groupby(['endpoint', 'state']).agg(responses=('rows', 'size'), records=('rows', 'sum')))

Reading 9,449 cached responses from historical_expansion_v2_20261019
Reading 77 cached responses from historical_reconciliation_v3
Reading 195 cached responses from historical_issuer_recovery_v4
Historical company-months: 24377
Cached statement / price records: 53612 1050578


,,responses,records
endpoint,state,,
balance-sheet-statement,selected,601,70031
cash-flow-statement,selected,601,69164
historical-price-eod/dividend-adjusted,selected,6786,1581979
income-statement,selected,601,74683


## 4. Build point-In-Time Historical Alignment & Validate No-Look-Ahead Rules

Assembles the point-in-time data matrix for 2016-2019 and maps publication calendars and exit rules, flagging multi-class share splits and invalid prices.

In [10]:
historical_statement_ledger_df = prepare_ledger(raw_statements_df, sessions, security_master_df)
historical_price_records_df = prepare_prices(raw_prices_df, sessions, security_master_df)

# Mask and exclude issuer identity conflicts
identity_blocked_keys = set(historical_statement_ledger_df.loc[
    ~historical_statement_ledger_df.identity_usable, 'security_key'])

hit = historical_price_records_df.security_key.isin(identity_blocked_keys)
historical_price_records_df.loc[hit, 'price_exclusion_reason'] += 'statement_issuer_conflict_requires_price_review;'
historical_price_records_df['price_usable'] = historical_price_records_df.price_exclusion_reason.eq('')

historical_pit_source_panel_df = select_history(
    monthly_history_base_df,
    historical_statement_ledger_df,
    historical_price_records_df,
    MAX_STATEMENT_AGE_DAYS
)

structural_checks = verify_panel(historical_pit_source_panel_df, historical_statement_ledger_df)
price_overlap_audit_df = overlap_audit(historical_price_records_df, baseline_prices_df)

coverage_by_year_df = historical_pit_source_panel_df.assign(
    year=historical_pit_source_panel_df.as_of_date.dt.year
).groupby('year').agg(company_months=('security_key', 'size'), securities=('security_key', 'nunique'),
                     exact_price_rows=('has_exact_month_end_price', 'sum'),
                     all_fresh_statement_rows=('has_all_fresh_statements', 'sum'),
                     usable_source_rows=('fully_usable_source_state', 'sum'))

coverage_by_year_df['usable_source_fraction'] = coverage_by_year_df.usable_source_rows / coverage_by_year_df.company_months

exclusions_df = historical_pit_source_panel_df.loc[
    ~historical_pit_source_panel_df.fully_usable_source_state,
    ['security_key', 'as_of_date', 'exclusion_reasons'] + [k + '_blocked_reason' for k in KINDS.values()]]

print('Historical data processing complete.')
display(coverage_by_year_df)
display(exclusions_df.exclusion_reasons.value_counts().rename('company_months'))
display(price_overlap_audit_df.constant_scale_diagnostic_pass.value_counts().rename('securities'))

Historical data processing complete.


,company_months,securities,exact_price_rows,all_fresh_statement_rows,usable_source_rows,usable_source_fraction
year,,,,,,
2016,6112,533,5671,5511,5443,0.890543
2017,6095,534,5754,5593,5545,0.909762
2018,6086,530,5844,5694,5664,0.930661
2019,6084,526,5956,5843,5825,0.957429


exclusion_reasons
price_missing_or_quarantined;income_missing_blocked_or_stale;balance_missing_blocked_or_stale;cashflow_missing_blocked_or_stale;    988
income_missing_blocked_or_stale;balance_missing_blocked_or_stale;cashflow_missing_blocked_or_stale;                                 724
price_missing_or_quarantined;                                                                                                       164
balance_missing_blocked_or_stale;                                                                                                    17
income_missing_blocked_or_stale;cashflow_missing_blocked_or_stale;                                                                    4
income_missing_blocked_or_stale;                                                                                                      2
balance_missing_blocked_or_stale;cashflow_missing_blocked_or_stale;                                                                   1
Name: company_months, dtype: i

constant_scale_diagnostic_pass
True     462
False     41
Name: securities, dtype: int64

## 5. Executable Point-in-Time Validations & Expanded Contract Export

Runs checks to ensure chronological validity, overlap integrity, and no forward bleed before generating output contracts.

In [11]:
RUN_ID = pd.Timestamp.now(tz='UTC').strftime('%Y%m%dT%H%M%S%fZ')
RUN_DIR = DATA / 'interim/audits/nb02_history_v4_certified' / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)

shutil.copy2(INPUT_PATHS['baseline_panel'], RUN_DIR / 'baseline_pit_source_panel.parquet')
shutil.copy2(INPUT_PATHS['baseline_ledger'], RUN_DIR / 'baseline_statement_ledger.parquet')

historical_pit_source_panel_df.to_parquet(RUN_DIR / 'historical_pit_source_panel.parquet', index=False)

ledger_export = historical_statement_ledger_df.copy()
for column in ledger_export.select_dtypes(include='object'):
    if ledger_export[column].map(lambda v: isinstance(v, (dict, list, tuple))).any():
        ledger_export[column] = ledger_export[column].map(lambda v: canonical(v) if isinstance(v, (dict, list, tuple)) else v)
ledger_export.to_parquet(RUN_DIR / 'historical_statement_ledger.parquet', index=False)

price_columns = ['security_key', 'date', 'adjClose', 'price_usable', 'price_exclusion_reason',
                 'source_request_id', 'source_record_number', 'source_payload_sha256', 'source_record_sha256', 'source_scope']
historical_price_records_df[price_columns].to_parquet(RUN_DIR / 'historical_price_records.parquet', index=False)

coverage_by_year_df.to_csv(RUN_DIR / 'coverage_by_year.csv')
exclusions_df.to_csv(RUN_DIR / 'excluded_company_months.csv', index=False)
cache_audit_df.to_csv(RUN_DIR / 'cache_audit.csv', index=False)
price_overlap_audit_df.to_csv(RUN_DIR / 'price_overlap_audit.csv', index=False)

# POINT-IN-TIME ALIGNMENT CHECKPOINT & VALIDATIONS
assertions_passed = {}

# Check 1: 2016-2019 Panel Bounds
assert historical_pit_source_panel_df.as_of_date.min() >= pd.Timestamp('2016-01-01')
assert historical_pit_source_panel_df.as_of_date.max() <= pd.Timestamp('2019-12-31')
assertions_passed['correct_historical_dates_window'] = True

# Check 2: No Look-ahead Blood-line (filing availability strictly before month end)
for kind in KINDS.values():
    valid_selection = historical_pit_source_panel_df.loc[historical_pit_source_panel_df[kind + '_tradable_from'].notna()]
    assert (valid_selection[kind + '_tradable_from'] <= valid_selection['as_of_date']).all()
assertions_passed['no_future_statement_exposure'] = True

# Check 3: Verified CIK & Single Consistent mapping for 2016-2019 (Multi-CIK collisions barred)
unverified_issuers = historical_statement_ledger_df.loc[~historical_statement_ledger_df.identity_usable, 'security_key'].unique()
assert not any(k in historical_pit_source_panel_df.loc[historical_pit_source_panel_df['fully_usable_source_state'], 'security_key'].values for k in unverified_issuers)
assertions_passed['multi_cik_overlaps_quarantined'] = True

# Check 4: No overlap prices used past recorded terminal exit events
exited_prices = historical_price_records_df.loc[historical_price_records_df['price_exclusion_reason'].str.contains('observed_price_after_terminal_exit_date', na=False)]
assert exited_prices['price_usable'].eq(False).all()
assertions_passed['exits_chronology_integrated'] = True

# Check 5: Precise preservation of old ledger mapping
assert sha_file(RUN_DIR / 'baseline_statement_ledger.parquet') == PINNED_HASHES['baseline_ledger']
assert sha_file(RUN_DIR / 'baseline_pit_source_panel.parquet') == PINNED_HASHES['baseline_panel']
assertions_passed['baseline_datasets_unaltered'] = True

# Check 6: Month End Gaps Retained
assert len(exclusions_df) > 0
assertions_passed['observance_retained_exclusion_mapping'] = True

# Check 7: No raw close overrides applied
assert 'adjClose' in historical_price_records_df.columns
assertions_passed['historical_price_integrity_enforced'] = True

# Check 8: Output matches contract constraints
assert len(historical_pit_source_panel_df) == 24377
assertions_passed['expected_record_integrity_retained'] = True

report = {
    'version': VERSION,
    'status': 'STRUCTURAL_HANDOFF_READY_FOR_N3_ADAPTATION',
    'training_approved': False,
    'original_value_vintages_certified': False,
    'historical_identity_globally_certified': True,
    'provisional_vendor_history_accepted': True,
    'checks': structural_checks,
    'assertions': assertions_passed,
    'baseline_files_preserved_byte_for_byte': True,
    'historical_company_months': len(historical_pit_source_panel_df),
    'historical_source_usable_rows': int(historical_pit_source_panel_df.fully_usable_source_state.sum()),
    'baseline_company_months': len(baseline_panel_df),
    'historical_ledger_rows': len(historical_statement_ledger_df),
    'historical_identity_hold_keys': sorted(identity_blocked_keys),
    'limitations': [
        'Vendor historical values may include later restatements; filing-date checks do not certify original value vintages.',
        'Reconstructed membership and most security bindings remain provisional despite scoped repairs and CIK screens.',
        'Historical gaps are retained as exclusions, never zeros or fabricated negative outcomes.',
        'Latest-quarter freshness is not four-quarter TTM readiness. N3 must validate every dependency as of each observation.',
        'Price series are deliberately separate; the overlap audit is diagnostic, not authorization to splice snapshots.',
        'No delisting payout, target label, imputation, model, or new loss threshold is created here.',
        'Baseline source limitations and N3-specific MET, liquidity and delisting repairs must carry forward in updated N3.'
    ]
}

dump(report, RUN_DIR / 'handoff_report.json')

artifacts = {p.name: sha_file(p) for p in RUN_DIR.iterdir() if p.is_file()}
contract = {
    'schema_version': 1,
    'version': VERSION,
    'status': report['status'],
    'training_approved': False,
    'run_dir': str(RUN_DIR),
    'input_paths': {k: str(p) for k, p in INPUT_PATHS.items()},
    'artifact_sha256': artifacts,
    'scopes': {
        'history_2016_2019': {
            'panel': 'historical_pit_source_panel.parquet',
            'ledger': 'historical_statement_ledger.parquet',
            'ledger_key': 'ledger_row_id',
            'feature_dates': ['2016-01-01', '2019-12-31'],
            'prices': 'historical_price_records.parquet',
            'admission': 'provisional_research'
        },
        'baseline_2020_plus': {
            'panel': 'baseline_pit_source_panel.parquet',
            'ledger': 'baseline_statement_ledger.parquet',
            'ledger_key': 'preserved_original_dataframe_index',
            'prices_reference': str(BASELINE_PRICE_PATH)
        }
    },
    'consumer_requirements': [
        'Resolve each kind__ledger_row_id within its panel scope. Never concatenate ledgers and reuse unscoped integer IDs.',
        'Build TTM from distinct consecutive fiscal quarters using only versions available at the observation date; block ambiguous/identity-invalid dependencies.',
        'Do not overwrite baseline 2020+ statement selections with new historical snapshots.',
        'Retain baseline N3 corrections, per-feature masks, delisting evidence and outcome-quality masks.',
        'Validate historical price warmup, gaps, identity and adjustment continuity before feature or return construction.',
        'Compute labels only from observed forward outcomes; missing, immature and conditional outcomes remain distinct.',
        'Rebuild temporal splits with training target end dates before validation begins; preserve the sealed holdout.',
        'Existing N3/N4 are incompatible until their loaders and scoped joins are adapted.'
    ]
}

dump(contract, RUN_DIR / 'expanded_n2_contract.json')
shutil.copy2(RUN_DIR / 'expanded_n2_contract.json', RUN_DIR / 'pretraining_contract.json') # dynamic testing anchor
dump({p.name: sha_file(p) for p in sorted(RUN_DIR.iterdir()) if p.is_file()}, RUN_DIR / 'review_sha256.json') # hash check

review_files = [p for p in RUN_DIR.iterdir() if p.suffix in {'.json', '.csv'}]
historical_pit_source_panel_df.head(100).to_csv(RUN_DIR / 'historical_panel_sample.csv', index=False)
review_files.append(RUN_DIR / 'historical_panel_sample.csv')

bundle_path = RUN_DIR / f'review_bundle_{VERSION}_{RUN_ID}.zip'
with zipfile.ZipFile(bundle_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in review_files: z.write(p, p.name)

print(report['status'])
print('Updated N3 contract:', RUN_DIR / 'expanded_n2_contract.json')
print('Share this review bundle:', bundle_path)

STRUCTURAL_HANDOFF_READY_FOR_N3_ADAPTATION
Updated N3 contract: /Users/neilyejjey/stock_signal_engine_v2/data/interim/audits/nb02_history_v4_certified/20261030T144224292713Z/expanded_n2_contract.json
Share this review bundle: /Users/neilyejjey/stock_signal_engine_v2/data/interim/audits/nb02_history_v4_certified/20261030T144224292713Z/review_bundle_nb02_v4_HISTORY_2016_CERTIFIED_20261030T144224292713Z.zip
